### Live Telemetry Data Ingestion & JSON Log Streaming 

This script sets up a lightweight background listener that acts as a secure gate for incoming server data on Port 8080. It catches incoming alerts, processes them instantly in memory, and logs them safely to your hard drive.

*   **Continuous Background Listening:** It runs quietly in its own dedicated lane, capturing incoming server traffic without freezing or slowing down your user interface.
*   **Automatic File Management:** Every time a log file hits a specific block limit, the code automatically creates a new file to save your data seamlessly without dropping any lines.
*   **Built-In Computer Memory Shields:** It tracks your laptop's memory use dynamically, cleaning up duplicate records and managing active row counts so it never uses more than 2GB of RAM.



In [1]:
import pandas as pd
import json
import os
import sys
from http.server import BaseHTTPRequestHandler, HTTPServer
import threading

# ==============================================================================
# CONFIGURATION PARAMETERS & DATA POOL INITIALIZATION
# ==============================================================================
FILE_PREFIX = "wzuh_siem_logs"
MEMORY_ROW_CEILING = 250000            # Safely caps active browser RAM under 2GB
LINES_PER_FILE_CEILING = 100000        # Limits individual JSON file blocks to ~50MB

# This persistent table holds your real-time alerts right in the notebook memory
if 'realtime_alerts_df' not in locals():
    realtime_alerts_df = pd.DataFrame()
    print("[+] Initialised fresh master Pandas DataFrame.")

# Safe state tracker to manage sequential JSON file block naming metrics
state_tracker = {
    'file_index': 1,
    'current_file_lines': 0
}

# Scan folder at startup to find the active uncompleted JSON log block
while True:
    filename = f"{FILE_PREFIX}_{state_tracker['file_index']}.json"
    if not os.path.exists(filename):
        break
    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
            line_count = sum(1 for _ in f)
        if line_count < LINES_PER_FILE_CEILING:
            state_tracker['current_file_lines'] = line_count
            break
    except Exception:
        break
    state_tracker['file_index'] += 1

# ==============================================================================
#  HTTP WEB CATCHER ( JSON STREAMING )
# ==============================================================================
class WazuhAlertCatcher(BaseHTTPRequestHandler):
    def do_POST(self):
        global realtime_alerts_df
        
        # 1. Read the length of the incoming real-time JSON alert string
        content_length = int(self.headers['Content-Length'])
        post_data = self.rfile.read(content_length).decode('utf-8')
        
        try:
            # 2. Parse the text payload directly into a Python dictionary mapping
            alert_json = json.loads(post_data)
            
            # CONTROL 2: Enforce 50MB Sequential Disk Block Rotation
            if state_tracker['current_file_lines'] >= LINES_PER_FILE_CEILING:
                state_tracker['file_index'] += 1
                state_tracker['current_file_lines'] = 0
                
            target_file = f"{FILE_PREFIX}_{state_tracker['file_index']}.json"
            
            # Save raw data immediately as a self-contained text line string object
            with open(target_file, 'a', encoding='utf-8', errors='ignore') as f:
                f.write(json.dumps(alert_json) + '\n')
            state_tracker['current_file_lines'] += 1
            
            # 3. Use json_normalize to flatten keys out for live notebook RAM table view
            new_row_df = pd.json_normalize(alert_json)
            realtime_alerts_df = pd.concat([realtime_alerts_df, new_row_df], ignore_index=True)
            
            # 4. Clean up duplicates dynamically using Wazuh's unique event identifier
            id_column = 'id' if 'id' in realtime_alerts_df.columns else '_id' if '_id' in realtime_alerts_df.columns else None
            if id_column:
                realtime_alerts_df.drop_duplicates(subset=[id_column], keep='first', inplace=True)
                realtime_alerts_df.reset_index(drop=True, inplace=True)
                
            # CONTROL 1: Enforce 2GB RAM Round-Robin Memory Cap
            if len(realtime_alerts_df) > MEMORY_ROW_CEILING:
                eviction_count = len(realtime_alerts_df) - MEMORY_ROW_CEILING
                realtime_alerts_df = realtime_alerts_df.iloc[eviction_count:].reset_index(drop=True)
            
            # 5. Flash an update string right underneath your live cell view
            print(f"[+] Alert Caught! Master Table Size: {len(realtime_alerts_df)} rows | Active Block: {target_file}", end="\r")
            
            # 6. Reply back to the Ubuntu container with a clean HTTP 200 OK success ticket
            self.send_response(200)
            self.send_header('Content-type', 'application/json')
            self.end_headers()
            self.wfile.write(b'{"status": "success"}')
            
        except Exception as e:
            try:
                self.send_response(500)
                self.end_headers()
            except:
                pass
            print(f"\n[-] Data Processing Error: {str(e)}")

    def log_message(self, format, *args):
        # Suppress normal web server background log clutter inside your notebook output window
        return

# ==============================================================================
# Activate PORT Listener 8080
# ==============================================================================
def run_server():
    server_address = ('', 8080) # Listens on all active Windows network card interfaces
    try:
        httpd = HTTPServer(server_address, WazuhAlertCatcher)
        print("[*] Engine Active! Listening on port 8080 for real-time JSON strings...")
        httpd.serve_forever()
    except Exception as network_error:
        sys.stderr.write(f"\n[-] Network bind conflict: {str(network_error)}\n")

# Fire the listener up inside a background thread so it doesn't freeze your notebook interface
listener_thread = threading.Thread(target=run_server, daemon=True)
listener_thread.start()



[+] Initialised fresh master Pandas DataFrame.


[*] Engine Active! Listening on port 8080 for real-time JSON strings...
